In [ ]:
# 1. 최신 생성 + 기존 Qwen3-VL 시각 근거 통합 실험 환경 확인
# 이 노트북은 GCP 레포 루트에 업로드하고 myenv 커널에서 실행합니다.
import getpass
import os
from pathlib import Path

ROOT = Path('/home/kongseok/sprint-public-procurement-rag-assistant')
assert ROOT.exists(), f'레포 경로가 없습니다: {ROOT}'
os.chdir(ROOT)

# 생성은 OpenAI API를 사용하므로 검색용 KURE는 CPU에서 실행합니다.
# 반드시 torch/sentence-transformers를 불러오기 전에 설정해야 합니다.
os.environ['CUDA_VISIBLE_DEVICES'] = ''

saved_key = os.environ.get('OPENAI_API_KEY', '').strip()
if not saved_key.isascii() or not saved_key.startswith('sk-'):
    saved_key = getpass.getpass('OpenAI API key: ').strip()
    os.environ['OPENAI_API_KEY'] = saved_key
assert saved_key.isascii() and saved_key.startswith('sk-'), '실제 OpenAI API 키를 입력하세요.'

from openai import OpenAI
base_client = OpenAI(api_key=saved_key)
base_client.models.list()
print('작업 위치:', Path.cwd())
print('OpenAI API 연결 확인 완료')

In [ ]:
# 2. 데이터, 최신 생성 코드, 기존 Qwen3-VL 시각 근거, 채점기 v3.0.1 로드
import importlib.util
import json
import re
import time
from types import SimpleNamespace

import pandas as pd

required = [
    ROOT / 'output/chunks.pkl',
    ROOT / 'output/chroma_db',
    ROOT / 'output/merged_docs.pkl',
    ROOT / 'data/golden_set_v3/rag-56.draft.jsonl',
    ROOT / 'data/golden_set_v3/set-13.draft.jsonl',
    ROOT / 'data/golden_set_v3/document-structure-visual-qa.jsonl',
    ROOT / 'src/evaluation/scoring_v3/scorer.py',
    ROOT / 'src/evaluation/scoring_v3/_base.py',
    ROOT / 'experiments/dahye_latest_20260909/answer_generation.py',
    ROOT / 'experiments/dahye_latest_20260909/generation_prompts.py',
    ROOT / 'output/experiments/b_plan_v3_vlm/visual_evidence.jsonl',
]
missing = [str(path) for path in required if not path.exists()]
assert not missing, '없는 파일:\n- ' + '\n- '.join(missing)

from experiments.dahye_latest_20260909.answer_generation import ask_rfp_v9
from src.data_processing.chunking import load_chunks
from src.evaluation.golden_set_v3 import load_golden_set_v3
from src.retrieval.embeddings import SentenceTransformerEmbedding
from src.retrieval.indexing import HybridIndex

scorer_path = ROOT / 'src/evaluation/scoring_v3/scorer.py'
spec = importlib.util.spec_from_file_location('simple_scorer_v3', scorer_path)
scorer = importlib.util.module_from_spec(spec)
spec.loader.exec_module(scorer)
assert scorer.SCORER_VERSION == '3.0.1', scorer.SCORER_VERSION

evidence_path = ROOT / 'output/experiments/b_plan_v3_vlm/visual_evidence.jsonl'
with evidence_path.open(encoding='utf-8') as handle:
    evidence_rows = [json.loads(line) for line in handle if line.strip()]
evidence_by_case = {row['case_id']: row for row in evidence_rows}
assert len(evidence_by_case) == 4, f'VLM 그림 근거가 4건이 아닙니다: {len(evidence_by_case)}'
assert all(not row.get('error') and row.get('evidence_text') for row in evidence_rows), 'VLM 근거에 오류 또는 빈 값이 있습니다.'
assert all(row.get('model') == 'Qwen/Qwen3-VL-8B-Instruct' for row in evidence_rows), '예상한 Qwen3-VL 근거가 아닙니다.'
print('검증된 Qwen3-VL 그림 근거:', len(evidence_by_case), '건')

chunks = load_chunks()
assert chunks, 'output/chunks.pkl을 읽지 못했습니다.'
child_chunks = [c for c in chunks if getattr(c, 'strategy', '') != 'parent']
corpus_doc_ids = {c.doc_id for c in chunks}
golden = load_golden_set_v3(corpus_doc_ids=corpus_doc_ids)
golden_rows = golden.to_dict('records')

doc_to_biz = {}
for chunk in chunks:
    meta = getattr(chunk, 'metadata', {}) or {}
    biz = meta.get('사업명') or meta.get('사업_명') or ''
    doc_to_biz.setdefault(chunk.doc_id, str(biz))
all_filenames_with_biz = sorted(doc_to_biz.items())

print(f'chunk {len(chunks):,}개, 문서 {len(corpus_doc_ids)}개, 골든셋 {len(golden_rows)}문항')
print('검색용 KURE를 CPU에 로드합니다.')
embedding_backend = SentenceTransformerEmbedding()
assert embedding_backend.name == 'nlpai-lab/KURE-v1', embedding_backend.name
index = HybridIndex(chunks, persist=True, embedding_backend=embedding_backend)
print('KURE-Chroma 검색기 준비 완료')

In [ ]:
# 3. VLM 시각 근거 주입, 기록 및 채점 보조 함수
class RecordingCompletions:
    def __init__(self, delegate, evidence=None):
        self.delegate = delegate
        self.evidence = evidence or {}
        self.last_prompt = ''

    def create(self, **kwargs):
        messages = [dict(message) for message in (kwargs.get('messages') or [])]
        evidence_text = str(self.evidence.get('evidence_text') or '')
        if messages and evidence_text:
            prompt = str(messages[-1].get('content', ''))
            doc_id = str(self.evidence.get('doc_id') or '')
            evidence_id = str(self.evidence.get('evidence_id') or 'visual-evidence')
            visual_block = (
                f'[문서: {doc_id}]\n'
                f'[Qwen3-VL 검증 시각 근거: {evidence_id}]\n'
                f'{evidence_text}\n\n'
            )
            prompt = prompt.replace('## 질문', visual_block + '## 질문', 1)
            messages[-1]['content'] = prompt
            kwargs['messages'] = messages
        self.last_prompt = str(messages[-1].get('content', '')) if messages else ''
        return self.delegate.create(**kwargs)

class RecordingOpenAI:
    def __init__(self, client, evidence=None):
        self.completions = RecordingCompletions(client.chat.completions, evidence)
        self.chat = SimpleNamespace(completions=self.completions)

    def reset(self):
        self.completions.last_prompt = ''

def client_for_case(case_id):
    return RecordingOpenAI(base_client, evidence_by_case.get(str(case_id)))

def extract_context(prompt):
    marker = '## 컨텍스트 (검색된 문서 조각)'
    if marker not in prompt:
        return ''
    return prompt.split(marker, 1)[1].split('## 질문', 1)[0].strip()

def cited_docs(answer):
    match = re.search(r'\[\s*근거\s*:\s*(.+?)\]\s*$', str(answer), re.DOTALL)
    return [x.strip() for x in match.group(1).split(',') if x.strip()] if match else []

def context_docs(context):
    pattern = r'^\[문서:\s*(.*)\]\s*$'
    return list(dict.fromkeys(re.findall(pattern, context, flags=re.MULTILINE)))

def fact_coverage(row, context):
    groups = scorer._BASE.fact_groups(row)
    if not groups or not context:
        return None
    matched = sum(any(scorer.option_matches(context, option) for option in group) for group in groups)
    return matched / len(groups)

def recall(expected, retrieved):
    expected = set(expected or [])
    return len(expected & set(retrieved)) / len(expected) if expected else None

OUT_DIR = ROOT / 'output/dahye_latest_notebook_vlm_v1'
OUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT = OUT_DIR / 'inference.jsonl'
print('결과 폴더:', OUT_DIR)

In [ ]:
# 4. VLM 그림 문항 1건으로 실제 연결 시험
# 시각 근거가 프롬프트에 들어갔고 답변 생성까지 성공해야 전체 실행으로 넘어갑니다.
test_row = next(row for row in golden_rows if row['id'] == 'visual-hwp-figure-001')
test_client = client_for_case(test_row['id'])
test_answer = ask_rfp_v9(
    test_row['query'], test_client, index, child_chunks, all_filenames_with_biz,
    model_name='gpt-5-mini',
)
assert test_answer and test_answer != '(답변 생성 실패)', test_answer
assert '[Qwen3-VL 검증 시각 근거:' in test_client.completions.last_prompt, 'VLM 시각 근거가 프롬프트에 주입되지 않았습니다.'
print('시험 문항:', test_row['id'])
print(test_answer)
print('VLM 시각 근거 주입 및 1문항 생성 성공 - 다음 셀을 실행해도 됩니다.')

In [ ]:
# 5. 골든셋 79문항 생성
# 성공 문항은 즉시 저장되며, 이 셀을 다시 실행하면 성공 문항은 건너뜁니다.
def read_checkpoint():
    if not CHECKPOINT.exists():
        return []
    with CHECKPOINT.open(encoding='utf-8') as handle:
        return [json.loads(line) for line in handle if line.strip()]

prior = read_checkpoint()
completed = {str(r['id']) for r in prior if r.get('id') and not r.get('execution_error')}
print(f'기존 성공 {len(completed)}건, 남은 {len(golden_rows) - len(completed)}건')

for number, row in enumerate(golden_rows, start=1):
    case_id = str(row['id'])
    if case_id in completed:
        continue
    started = time.perf_counter()
    recording_client = client_for_case(case_id)
    answer = None
    error = None
    try:
        answer = ask_rfp_v9(
            row['query'], recording_client, index, child_chunks, all_filenames_with_biz,
            model_name='gpt-5-mini',
        )
        if answer == '(답변 생성 실패)':
            error = 'answer_generation_returned_failure'
    except Exception as exc:
        error = f'{type(exc).__name__}: {exc}'

    context = extract_context(recording_client.completions.last_prompt)
    retrieved = context_docs(context)
    if not retrieved and answer:
        retrieved = cited_docs(answer)
    result = {
        'id': case_id,
        'answer': answer,
        'execution_error': error,
        'retrieved_doc_ids': retrieved,
        'retrieval_recall': recall(row.get('expected_doc_id'), retrieved),
        'context_fact_coverage': fact_coverage(row, context),
        'elapsed_seconds': round(time.perf_counter() - started, 3),
        'visual_evidence_used': case_id in evidence_by_case,
        'visual_evidence_id': (evidence_by_case.get(case_id) or {}).get('evidence_id'),
        'visual_model': (evidence_by_case.get(case_id) or {}).get('model'),
    }
    with CHECKPOINT.open('a', encoding='utf-8') as handle:
        handle.write(json.dumps(result, ensure_ascii=False) + '\n')
    if error:
        raise RuntimeError(f'{case_id} 생성 실패: {error}. 오류를 고친 뒤 이 셀을 다시 실행하세요.')
    completed.add(case_id)
    print(f'[{number:02d}/{len(golden_rows)}] {case_id} 완료')

print('79문항 생성 완료')

In [ ]:
# 6. 채점기 v3.0.1로 최종 채점
latest = {}
for row in read_checkpoint():
    case_id = str(row.get('id', ''))
    if case_id and (case_id not in latest or not row.get('execution_error')):
        latest[case_id] = row
predictions = [latest[str(row['id'])] for row in golden_rows if str(row['id']) in latest]
assert len(predictions) == 79, f'완료 문항이 79개가 아닙니다: {len(predictions)}'
assert not any(row.get('execution_error') for row in predictions), '실행 오류 문항이 남아 있습니다.'

details, summary = scorer.evaluate(golden_rows, predictions)
prediction_by_id = {str(row['id']): row for row in predictions}
golden_by_id = {str(row['id']): row for row in golden_rows}
for detail in details:
    case_id = str(detail['id'])
    prediction = prediction_by_id[case_id]
    detail['source_lane'] = golden_by_id[case_id].get('source_lane')
    detail['visual_evidence_used'] = bool(prediction.get('visual_evidence_used'))
    detail['visual_evidence_id'] = prediction.get('visual_evidence_id')
    detail['visual_model'] = prediction.get('visual_model')

def average(rows, field):
    values = [float(row[field]) for row in rows if row.get(field) is not None]
    return sum(values) / len(values) if values else None

visual_details = [row for row in details if row.get('source_lane') == 'visual']
figure_details = [row for row in visual_details if row.get('visual_evidence_used')]
table_details = [row for row in visual_details if not row.get('visual_evidence_used')]
summary.update({
    'run_id': 'dahye-latest-notebook-vlm-v1',
    'generation_model': 'gpt-5-mini',
    'visual_model': 'Qwen/Qwen3-VL-8B-Instruct',
    'visual_evidence_case_count': 4,
    'visual_integration': 'verified saved VLM evidence injected for figure cases; structured text retained for table cases',
    'visual_evaluation': {
        'total_visual_cases': len(visual_details),
        'vlm_figure_cases': len(figure_details),
        'structured_table_cases': len(table_details),
        'visual_end_to_end_score': average(visual_details, 'end_to_end_score'),
        'vlm_figure_end_to_end_score': average(figure_details, 'end_to_end_score'),
        'structured_table_end_to_end_score': average(table_details, 'end_to_end_score'),
    },
    'generation_source_commit': '9c4429ee0963c76d1a08bfc06528e0c1349cb3c9',
    'output_directory': str(OUT_DIR),
})
scorer.write_jsonl(OUT_DIR / 'scored_details.jsonl', details)
pd.DataFrame(details).to_csv(OUT_DIR / 'scored_details.csv', index=False, encoding='utf-8-sig')
(OUT_DIR / 'summary.json').write_text(
    json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8'
)
print(json.dumps(summary, ensure_ascii=False, indent=2))
print('결과 폴더:', OUT_DIR)